# Stage 4 — ROI Zone Check (Post-processing)

**Purpose**: Validate that each part appears within the expected zone before reporting the defect classification result.

**Why ROI check?**  
A fixed camera monitors a workstation. Operators place parts manually — occasionally a part lands outside the expected zone. A misaligned part produces a feature distribution the classifier was never trained on, so the defect prediction would be unreliable. The ROI check is a hard gate that runs *before* trusting the classifier output.

**Pipeline position**: `preprocess() → classify() → ROI check → final verdict`

---

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

from src.preprocessing import preprocess, load_image
from src.postprocessing import (
    define_roi, find_part_bbox, check_roi, draw_overlay,
    DEFAULT_ROI_MARGIN, MIN_OVERLAP_RATIO
)

DATA_ROOT = Path('../data/mvtec_ad/metal_nut')
print('Imports OK')
print(f'Default ROI margin: {DEFAULT_ROI_MARGIN*100:.0f}% per side')
print(f'Min overlap to pass: {MIN_OVERLAP_RATIO*100:.0f}%')

## 1. ROI definition

The ROI is the central 60% of the image (20% margin on each side). On 224×224 images this corresponds to a 135×135 pixel zone.

In [ ]:
img_good   = preprocess(str(DATA_ROOT / 'train/good/000.png'))
img_defect = preprocess(str(DATA_ROOT / 'test/bent/000.png'))

roi = define_roi(img_good.shape)
rx1, ry1, rx2, ry2 = roi
print(f'Image size: {img_good.shape[:2]}')
print(f'ROI:        ({rx1},{ry1}) -> ({rx2},{ry2})  [{rx2-rx1}x{ry2-ry1} px]')

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(img_good)
rect = patches.Rectangle((rx1, ry1), rx2-rx1, ry2-ry1,
                           linewidth=2, edgecolor='yellow', facecolor='none')
ax.add_patch(rect)
ax.set_title('ROI definition (yellow = expected zone)')
ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Part localization via background thresholding

MVTec images have a near-black background. Thresholding at intensity=30 isolates the part as a bright foreground blob.

In [ ]:
bbox_good   = find_part_bbox(img_good)
bbox_defect = find_part_bbox(img_defect)

print(f'Good part bbox:     {bbox_good}')
print(f'Defective part bbox:{bbox_defect}')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, img, bbox, title in [
    (axes[0], img_good,   bbox_good,   'Good'),
    (axes[1], img_defect, bbox_defect, 'Defective (bent)'),
]:
    ax.imshow(img)
    if bbox:
        x1, y1, x2, y2 = bbox
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor='cyan', facecolor='none')
        ax.add_patch(rect)
    roi_rect = patches.Rectangle((rx1, ry1), rx2-rx1, ry2-ry1,
                                   linewidth=2, edgecolor='yellow', facecolor='none')
    ax.add_patch(roi_rect)
    ax.set_title(f'{title}\ncyan=part bbox, yellow=ROI')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. ROI check — normal cases

In [ ]:
for img, bbox, label, is_defective, prob in [
    (img_good,   bbox_good,   'Good (train)',      False, 0.03),
    (img_defect, bbox_defect, 'Defective (bent)',  True,  0.94),
]:
    in_roi, overlap = check_roi(bbox, roi)
    print(f'{label}: overlap={overlap:.2%}, in_roi={in_roi}')
    vis = draw_overlay(img, roi, bbox, in_roi, is_defective, prob)
    plt.figure(figsize=(4, 4))
    plt.imshow(vis); plt.title(label); plt.axis('off')
    plt.tight_layout(); plt.show()

## 4. Simulated out-of-position case

To demonstrate the out-of-position detection, we shift the image so the part appears near the corner — simulating a misplaced part on the workstation.

In [ ]:
import cv2

# Shift image 80px right and 80px down to simulate misplacement
M = np.float32([[1, 0, 80], [0, 1, 80]])
img_shifted = cv2.warpAffine(
    (img_good * 255).astype(np.uint8), M, (224, 224)
).astype(np.float32) / 255.0

bbox_shifted = find_part_bbox(img_shifted)
in_roi, overlap = check_roi(bbox_shifted, roi)

print(f'Shifted part bbox: {bbox_shifted}')
print(f'Overlap with ROI:  {overlap:.2%}')
print(f'ROI check passed:  {in_roi}  <-- OUT OF POSITION')

vis = draw_overlay(img_shifted, roi, bbox_shifted, in_roi, False, 0.1)
plt.figure(figsize=(5, 5))
plt.imshow(vis)
plt.title('Simulated out-of-position part\n(magenta = position alert)')
plt.axis('off')
plt.tight_layout()
plt.show()

## 5. Full pipeline — classify + ROI check combined

The final verdict combines both checks: a part must be **in position AND not defective** to pass.

In [ ]:
cases = [
    ('Good — in position',       img_good,    False, 0.03),
    ('Defective — in position',  img_defect,  True,  0.94),
    ('Good — OUT OF POSITION',   img_shifted, False, 0.10),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, img, is_defective, prob) in zip(axes, cases):
    bbox = find_part_bbox(img)
    in_roi, overlap = check_roi(bbox, roi) if bbox else (False, 0.0)
    vis  = draw_overlay(img, roi, bbox, in_roi, is_defective, prob)

    verdict = 'PASS' if (in_roi and not is_defective) else 'REJECT'
    reason  = 'out of position' if not in_roi else ('defective' if is_defective else '')
    ax.imshow(vis)
    ax.set_title(f'{title}\nVerdict: {verdict}' + (f' ({reason})' if reason else ''))
    ax.axis('off')

plt.suptitle('Full Pipeline — ROI + Defect Classification', fontsize=13)
plt.tight_layout()
plt.show()

## Summary — Design Decisions

| Decision | Choice | Why |
|---|---|---|
| ROI shape | Rectangle (central 60%) | Simple, interpretable, no annotation needed |
| Part localization | Background thresholding | MVTec has black background — reliable, fast, no ML needed |
| Overlap threshold | 80% | Tolerates slight misalignment; rejects clearly out-of-position parts |
| Pipeline order | ROI check after classification | Classification is fast; ROI check adds a spatial safety gate on the result |
| Color coding | Green=good, Red=defective, Magenta=out-of-position | Intuitive factory floor conventions |